# NB-24 — is a censored utterance resistant, or did the attack just fail?

**The review's sharpest methodological point.** 42% of our observations never
reach their target and are recorded as right-censored. The paper reads that as
resistance. It could equally be optimisation failure, and if the failure rate
differs between models it biases every comparison of \eps{} we make.

Three specific weaknesses in the attack, all from the review:

1. **Fixed learning rate.** Adam runs at `lr = 1e-4` at every budget from
   `1e-4` to `2e-3`. At the floor the step is as large as the whole budget.
2. **No restarts.** Each utterance gets one optimisation run from `delta = 0`.
3. **An absolute stall rule.** A probe is abandoned after 100 steps without the
   loss improving by `1e-3`. How often that fires depends on the model's loss
   scale, which is a run-level property.

**What this notebook does.** Takes the utterances that were censored on two
checkpoints, and attacks them again *much harder*:

| | published run | this run |
|---|---|---|
| precision | fp16 autocast | **fp32** |
| learning rate | fixed `1e-4` | **scaled to the budget** |
| restarts | none (`delta = 0`) | **5**, uniform in the eps-ball |
| steps | 400 probe / 800 confirm | **2,000** |
| stall rule | 100 steps | **off** |

It attacks at the **ceiling only** (`2e-3`). The question is binary: does this
utterance fall at the largest budget we allow, given a serious optimisation
effort? Every one that flips was a false negative in the published run.

**Reading the result.** If few flip, censoring means resistance and \eps{} is
sound. If many flip, the censoring rate is partly an artefact — and if the flip
rate differs between the 1-hour and 100-hour checkpoint, the comparison between
them is biased, not just noisy.

**Cost.** Only censored utterances, two checkpoints. Roughly 25 + 63 = 88
utterances at 5 restarts x 2,000 steps. Budget 4–6 GPU-hours.


## 1 · What to re-attack, and the deliberately stronger settings

In [ ]:
NOTEBOOK = "NB-24-attack-strength-check"

# One low-rung and one high-rung checkpoint, so the flip rate can be compared
# across the exposure range. If it differs, comparisons of eps50 between these
# two are biased and not merely noisy.
#
# `source_csv` is the published result file this reads the censored utt_ids
# from. Attach the dataset that carries it.
QUEUE = [
    dict(tag="rung1-final", ckpt_dataset="ckpt-xlsr-1h", rung=1,
         checkpoint="final", source_csv="eps50_rung1_final_T5_n100.csv"),
    dict(tag="rung100-ms3500", ckpt_dataset="ckpt-xlsr-100h", rung=100,
         checkpoint="milestone-3500",
         source_csv="eps50_rung100_milestone-3500_T5_n100.csv"),
]

WALL_HOURS = 11.0
MIN_HOURS_TO_START = 0.5

# --- unchanged, so the comparison is like-for-like -------------------------
EDIT_TYPE  = "T5"
N_EVAL     = 100
BATCH_SIZE = 4            # fp32 + 5 restarts needs more memory per sample
EPS_HI     = 2e-3         # ceiling only: this is a yes/no question
SEED       = 1337

# --- deliberately STRONGER than the published attack -----------------------
USE_AMP        = False    # fp32 throughout; no GradScaler skipped steps
STRONG_STEPS   = 2000     # against 400 probe / 800 confirm
N_RESTARTS     = 5        # against 1, and from random points in the eps-ball
LR_SCALE       = 0.05     # lr = LR_SCALE * eps. At the ceiling this is exactly
                          # 1e-4, the published LR, so nothing changes there and
                          # smaller budgets get a proportionally smaller step.
STALL_PATIENCE = 10 ** 9  # off
CHECK_EVERY    = 50

# These exist only because the cloned core cells read them.
EPS_LO, BISECT_ROUNDS, PROBE_STEPS, CONFIRM_STEPS = 1e-4, 0, STRONG_STEPS, STRONG_STEPS
FIXED_EPS, LR, VALIDATE_BATCHING = EPS_HI, LR_SCALE * EPS_HI, False

assert USE_AMP is False, "the whole point is fp32; AMP overflow skips steps for a whole batch"
assert N_RESTARTS >= 2 and STRONG_STEPS >= 2000
assert abs(LR_SCALE * EPS_HI - 1e-4) < 1e-12, (
    "LR_SCALE must reproduce the published 1e-4 at the ceiling, or a flip could "
    "be the learning rate rather than the extra effort.")

print(f"{NOTEBOOK} | {len(QUEUE)} checkpoints | ceiling {EPS_HI:.0e} only")
print(f"  fp32, {N_RESTARTS} restarts, {STRONG_STEPS} steps, no stall rule, "
      f"lr = {LR_SCALE} x eps")
print("\nA censored utterance that now succeeds was a FALSE NEGATIVE in the")
print("published run, not a resistant utterance.")

## 2 · Boilerplate

In [ ]:
import os, hashlib
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # attacks optimise the INPUT: never DataParallel
import sys, json, time, random, re, glob, unicodedata, gc
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
torch.backends.cudnn.benchmark = True

SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
WORK, TEMP, DEVICE = "/kaggle/working", "/kaggle/temp", "cuda"
assert torch.cuda.is_available(), "Settings -> Accelerator -> GPU T4 x2"

# PROV holds only what is defined AT THIS POINT. In NB-05 the rung, edit type
# and bracket were module-level constants, so they could be read here; in a
# queue runner they are per-condition and are not bound yet. Reading them here
# is a NameError that fires three cells before anything expensive starts. The
# run-specific fields are added by the summary cell at the end.
# Project-wide Bengali normaliser. Defined HERE, in boilerplate, because the
# input-resolution cell asserts with it and in NB-05 it lived in the (later)
# target-construction cell -- which a queue runner does not have.
ZW = dict.fromkeys(map(ord, "\u200c\u200d"), None)
def norm_bn(s):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", str(s)).translate(ZW)).strip()

PROV = {"notebook": NOTEBOOK, "seed": SEED,
        "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}
print(json.dumps(PROV, indent=2))

In [ ]:
# Internet is off in this session, so `pip install transformers==4.44.2` cannot
# reach the index. Measured twice in this project's own logs (NB-12 and NB-13,
# 2026-09-30): the cell spends ~202 s timing out, prints two ERROR lines, fails,
# and leaves the preinstalled version in place anyway. Pure waste.
#
# So we do not attempt the pin. We import what is already present and RECORD the
# version instead -- the pin's purpose was reproducibility, and a recorded
# version serves that better than a failed install does.
#
# If internet is ever turned ON, restore the pin: 4.44.2 is the version every
# figure measured before 2026-09-30 was produced on, and mixing versions across
# figures is exactly what the provenance record exists to catch.
#
# `jiwer` is deliberately NOT imported: it is not preinstalled on Kaggle, and
# the only place this notebook would use it is a FIXED_EPS clean-WER branch
# that cannot execute (FIXED_EPS is None here) and already guards itself.
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import transformers, librosa, numpy
print(f"transformers {transformers.__version__} | numpy {numpy.__version__} "
      f"| librosa {librosa.__version__}")
if transformers.__version__ != "4.44.2":
    print(f"  NOTE: this is NOT the project pin (4.44.2). Record "
          f"{transformers.__version__} against every number this run produces.")
PROV["transformers"] = transformers.__version__
PROV["numpy"] = numpy.__version__


## 3 · Inputs, eval set and frozen targets

In [ ]:
# ---------------------------------------------------------------------------
# Kaggle does not use one mount layout. Depending on the kernel you get
#     /kaggle/input/<name>                (older kernels)
#     /kaggle/input/datasets/<name>       (your own datasets, current)
#     /kaggle/input/competitions/<name>   (competition data)
# A flat glob only sees the wrapper dirs -- NB-01 failed with exactly this,
# reporting: Visible inputs: ['competitions', 'datasets']
# ---------------------------------------------------------------------------
inputs = sorted({p for pat in ("/kaggle/input/*", "/kaggle/input/*/*",
                               "/kaggle/input/*/*/*")
                 for p in glob.glob(pat) if os.path.isdir(p)})

SPLITS = next((p for p in inputs if "splits" in p.lower()), None)
assert SPLITS, ("Attach bn-asr-splits -- test.csv is needed to resolve audio paths "
                "for the 100 target utterances.\n"
                f"Visible inputs: {[os.path.basename(p) for p in inputs]}")
print("splits:", SPLITS)

# ---------------------------------------------------------------------------
# EVAL SET AND TARGETS: read, never regenerated.
#
# Task B's targets are published inside every eps50-*-n100 dataset as
# task_b_targets_n100.csv. Reading them makes this run byte-comparable by
# construction rather than by reproducing a generator. The hash is checked.
# ---------------------------------------------------------------------------
TGT_CSV = next((f"{p}/task_b_targets_n100.csv" for p in inputs
                if os.path.exists(f"{p}/task_b_targets_n100.csv")), None)
assert TGT_CSV, ("task_b_targets_n100.csv not found. Attach ONE Task B result "
                 "dataset, e.g. zoayriaabedin/eps50-rung1-final-t5-n100.\n"
                 f"Visible inputs: {[os.path.basename(p) for p in inputs]}")
TGT_SHA = hashlib.sha256(open(TGT_CSV, "rb").read()).hexdigest()
print("targets:", TGT_CSV)
print("targets sha256:", TGT_SHA)
assert TGT_SHA == "8711dc56202b33f90e769c9735231b0fa8cf27fcbc20151f5466bcd9cc2b1256", (
    "These are NOT the targets the six Task B runs used. The comparison against "
    "run 1's 75% would be meaningless.")
tg = pd.read_csv(TGT_CSV)
assert len(tg) == 100 and tg.utt_id.nunique() == 100

_full = pd.read_csv(f"{SPLITS}/test.csv")
# source_tokens / target_tokens are carried through: the results writer uses
# both, and eps50_per_token divides by target_tokens. Dropping them here would
# KeyError at save time, after the whole sweep had run.
ev = tg[["utt_id", "target_text", "source_tokens", "target_tokens"]].merge(
        _full, on="utt_id", how="left")
assert ev["path"].notna().all(), "some target utterances are absent from test.csv"
assert (ev.text.map(norm_bn).values == tg.text.map(norm_bn).values).all(), \
    "transcript mismatch between the target file and test.csv"
EV_CSV = TGT_CSV
PROV["attack_eval_csv"] = os.path.basename(TGT_CSV)
PROV["targets_sha256"] = TGT_SHA
PROV["fixed_eps"] = FIXED_EPS
PROV["bisect_rounds"] = BISECT_ROUNDS
# The checkpoint dataset and model_dir are PER CONDITION here, so they are not
# recorded once at the top: every result row carries its own `model_dir`, and the
# queue summary records the tags.
for _c in ("path","text","target_text","source_tokens","target_tokens","duration"):
    assert _c in ev.columns, f"ev is missing {_c}, which a later cell needs"
print(f"eval set: {len(ev)} utterances, manifest order (original 25 first)")

# ---------------------------------------------------------------------------
# Re-resolve the audio root, same hazard as NB-01: NB-00 baked in the mount
# point from ITS session. Here the whole attack set is only 25 files, so check
# every one of them rather than sampling.
# ---------------------------------------------------------------------------
def resolve_audio(df):
    p0 = str(df["path"].iloc[0])
    if os.path.exists(p0):
        return df, os.path.dirname(p0)
    stem, ext = os.path.basename(os.path.dirname(p0)), os.path.splitext(p0)[1]
    cands = [d for pat in (f"/kaggle/input/*/{stem}",
                           f"/kaggle/input/*/*/{stem}",
                           f"/kaggle/input/*/*/*/{stem}")
             for d in glob.glob(pat) if os.path.isdir(d)]
    assert cands, (f"Audio directory '{stem}' not found under /kaggle/input.\n"
                   "Attach: + Add Input -> Competitions -> 'bengaliai speech'.\n"
                   f"Visible inputs: {[os.path.basename(p) for p in inputs]}")
    df = df.copy()
    df["path"] = cands[0] + "/" + df["utt_id"].astype(str) + ext
    return df, cands[0]

ev, AUDIO_ROOT = resolve_audio(ev)
absent = [p for p in ev["path"] if not os.path.exists(p)]
assert not absent, f"{len(absent)}/{len(ev)} attack audio files missing, e.g.\n  {absent[0]}"
print(f"audio root: {AUDIO_ROOT}")
print(f"attack set: {len(ev)} utterances, all present")

## 4 · Front end, loss, PGD, bisection

Byte-identical to NB-05. These read `model`, `processor`, `BLANK_ID` and `DO_NORMALIZE` as globals at call time, which is what lets Stage 1 swap checkpoints without redefining anything.

In [ ]:
SR = 16000

def load_batch(rows):
    """Length-bucketed batch. Returns padded waveforms + a validity mask."""
    wavs = []
    for _, r in rows.iterrows():
        a, _ = librosa.load(r["path"], sr=SR, mono=True)
        a = np.asarray(a, np.float32)
        pk = np.abs(a).max()
        wavs.append(a / max(pk, 1.0))
    T = max(len(w) for w in wavs)
    x = torch.zeros(len(wavs), T, dtype=torch.float32)
    m = torch.zeros(len(wavs), T, dtype=torch.float32)
    for i, w in enumerate(wavs):
        x[i, :len(w)] = torch.from_numpy(w); m[i, :len(w)] = 1.0
    return x.to(DEVICE), m.to(DEVICE)


def frontend_b(x, mask):
    """Masked zero-mean/unit-variance. Computing stats over the PADDING would
    corrupt short utterances in a mixed-length batch."""
    if DO_NORMALIZE:
        n = mask.sum(-1, keepdim=True).clamp(min=1)
        mu = (x * mask).sum(-1, keepdim=True) / n
        var = (((x - mu) * mask) ** 2).sum(-1, keepdim=True) / n
        x = (x - mu) / torch.sqrt(var + 1e-7)
        x = x * mask
    return x


def logits_b(x, mask):
    return model(input_values=frontend_b(x, mask),
                 attention_mask=mask.long()).logits


def ctc_per_sample(x, mask, tgt_pad, tgt_len):
    """Per-sample CTC loss -> lets us mask out finished samples."""
    lg = logits_b(x, mask)
    logp = F.log_softmax(lg.float(), -1).transpose(0, 1)          # (L,B,V)
    in_len = model._get_feat_extract_output_lengths(mask.sum(-1).long()).long()
    in_len = in_len.clamp(max=logp.shape[0])
    return F.ctc_loss(logp, tgt_pad, in_len, tgt_len,
                      blank=BLANK_ID, zero_infinity=True, reduction="none")


def decode_b(x, mask):
    with torch.no_grad():
        ids = torch.argmax(logits_b(x, mask), -1)
    return [norm_bn(s) for s in processor.batch_decode(ids)]


def int16_rt(x):
    return torch.clamp((x * 32767.0).round(), -32768.0, 32767.0) / 32767.0

def snr_db_b(clean, adv, mask):
    n = (adv - clean) * mask
    s = clean * mask
    return (10 * torch.log10(s.pow(2).sum(-1) / (n.pow(2).sum(-1) + 1e-12))).cpu().numpy()

In [ ]:
def pgd_batch(x, mask, tgt_pad, tgt_len, targets, eps_vec,
              steps, delta_init=None, lr=LR):
    """eps_vec: (B,) per-sample L-inf budgets.
       Returns (success bool array, frozen deltas, steps_used)."""
    B = x.shape[0]
    eps_col = eps_vec.view(-1, 1)
    delta = (torch.zeros_like(x) if delta_init is None
             else torch.max(torch.min(delta_init.clone(), eps_col), -eps_col))
    delta.requires_grad_(True)
    opt = torch.optim.Adam([delta], lr=lr)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    active = torch.ones(B, dtype=torch.bool, device=DEVICE)
    frozen = torch.zeros_like(x)
    success = np.zeros(B, dtype=bool)
    best_loss = torch.full((B,), float("inf"), device=DEVICE)
    stall = torch.zeros(B, dtype=torch.long, device=DEVICE)

    for step in range(steps):
        if not active.any():
            break
        x_adv = torch.clamp(x + delta, -1, 1) * mask
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            per = ctc_per_sample(x_adv, mask, tgt_pad, tgt_len)
        loss = (per * active.float()).sum() / active.float().sum().clamp(min=1)

        opt.zero_grad(); scaler.scale(loss).backward()
        scaler.unscale_(opt)
        if delta.grad is not None:
            delta.grad[~active] = 0          # finished samples stop moving
        scaler.step(opt); scaler.update()
        with torch.no_grad():
            delta.data = torch.max(torch.min(delta.data, eps_col), -eps_col)

        with torch.no_grad():                # stall detection
            improved = per < best_loss - 1e-3
            best_loss = torch.minimum(best_loss, per.detach())
            stall = torch.where(improved, torch.zeros_like(stall), stall + 1)
            give_up = active & (stall > STALL_PATIENCE)
            active = active & ~give_up

        if (step + 1) % CHECK_EVERY == 0:
            hyp = decode_b(int16_rt(torch.clamp(x + delta, -1, 1)) * mask, mask)
            with torch.no_grad():
                for i in range(B):
                    if active[i] and hyp[i] == targets[i]:
                        frozen[i] = delta[i].detach()
                        success[i] = True
                        active[i] = False
    return success, frozen, step + 1

In [ ]:
def eps50_batch(x, mask, tgt_pad, tgt_len, targets, verbose=True):
    """Per-sample geometric bisection for the smallest budget that still works.

    ---------------------------------------------------------------------------
    BUG FIXED 2026-09-22 (found in the rung-1 T5 run, 40% censoring).

    The previous version ended with

        ok, best, _ = pgd_batch(..., hi, CONFIRM_STEPS, delta_init=best)

    which REASSIGNED `best` wholesale to pgd_batch's return value. That return
    is a `frozen` buffer holding zeros for any sample that did not succeed on
    that particular pass -- so one unlucky final pass wiped out a delta that had
    already been verified at a larger budget. Those samples were then written
    out as censored, with SNR 70-86 dB (the signature of delta == 0) even though
    the ceiling probe had already proved them attackable.

    In the rung-1 run that discarded ~6 of 25 real measurements. Worse, it does
    so at a RATE THAT VARIES BY MODEL, so comparing median eps50 across rungs
    would have been comparing differently-biased subsamples -- straight into
    RQ1's headline.

    Fix: keep a running `best_eps` / `best_delta` that is only ever updated on a
    VERIFIED success. Nothing overwrites a confirmed result. Censoring now means
    exactly one thing: the attack failed at every budget tried, ceiling included.

    Side benefit: eps50 becomes a guaranteed UPPER BOUND on the true minimum.
    Optimiser noise can only make it conservative, never wrong in an
    unpredictable direction -- which is the property you want in a metric you
    are about to compare across four models.
    ---------------------------------------------------------------------------
    """
    B = x.shape[0]
    lo = torch.full((B,), EPS_LO, device=DEVICE)
    hi = torch.full((B,), EPS_HI, device=DEVICE)
    best_eps   = torch.full((B,), float("nan"), device=DEVICE)   # verified only
    best_delta = torch.zeros_like(x)

    def record(okt, eps_vec, d):
        """Adopt a result only when it is both successful and better."""
        nonlocal best_eps, best_delta
        better = okt & (torch.isnan(best_eps) | (eps_vec < best_eps))
        best_eps   = torch.where(better, eps_vec, best_eps)
        best_delta = torch.where(better.view(-1, 1), d, best_delta)

    # Ceiling probe. Failure here is genuine right-censoring.
    ok, d, _ = pgd_batch(x, mask, tgt_pad, tgt_len, targets, hi, CONFIRM_STEPS)
    okt = torch.tensor(ok, device=DEVICE)
    record(okt, hi, d)
    feasible = ok.copy()
    if verbose:
        print(f"  ceiling probe eps={EPS_HI:.1e}: {feasible.sum()}/{B} feasible, "
              f"{(~feasible).sum()} censored")

    for rnd in range(BISECT_ROUNDS):
        mid = torch.sqrt(lo * hi)
        ok, d, _ = pgd_batch(x, mask, tgt_pad, tgt_len, targets, mid,
                             PROBE_STEPS, delta_init=best_delta)
        ok = ok & feasible
        okt = torch.tensor(ok, device=DEVICE)
        hi = torch.where(okt, mid, hi)
        lo = torch.where(okt, lo, mid)
        record(okt, mid, d)
        if verbose:
            print(f"  round {rnd+1}/{BISECT_ROUNDS}: {ok.sum()}/{B} succeeded "
                  f"| median eps {mid.median().item():.3e}")

    # Budget-consistency round: one more bisection at the FULL step budget, so
    # the reported boundary is measured at the budget eps50 is defined at rather
    # than at the cheaper probe budget. See RUNLOG for why this matters.
    #
    # Skipped in FIXED_EPS mode: there is no bracket to refine. The ceiling probe
    # already ran at CONFIRM_STEPS, so the single reported budget is measured at
    # the full budget and needs no second pass.
    if BISECT_ROUNDS == 0:
        eps       = best_eps.cpu().numpy()
        confirmed = ~np.isnan(eps)
        if verbose:
            print(f"  -> FIXED_EPS {EPS_HI:.1e}: {confirmed.sum()}/{B} succeeded, "
                  f"{(~confirmed).sum()} resisted")
        return eps, np.ones(B), confirmed, best_delta

    mid = torch.sqrt(lo * hi)
    ok, d, _ = pgd_batch(x, mask, tgt_pad, tgt_len, targets, mid,
                         CONFIRM_STEPS, delta_init=best_delta)
    ok = ok & feasible
    okt = torch.tensor(ok, device=DEVICE)
    record(okt, mid, d)
    if verbose:
        print(f"  budget-consistency round @ {CONFIRM_STEPS} steps: {ok.sum()}/{B}")

    eps       = best_eps.cpu().numpy()
    confirmed = ~np.isnan(eps)                 # censored == never succeeded
    width     = (hi / lo).cpu().numpy()
    if verbose:
        print(f"  -> {confirmed.sum()}/{B} measured, {(~confirmed).sum()} censored")
    return eps, width, confirmed, best_delta


## 5 · Per-condition loader

In [ ]:
def load_condition(cfg):
    """Resolve and load one checkpoint, rebinding the globals the attack closures read.

    The closures in the cells above look up `model`, `processor`, `BLANK_ID` and
    `DO_NORMALIZE` at CALL time, not definition time, so rebinding them here is
    all that is needed to point the whole machine at a different checkpoint.
    Every one of the four must be rebound: leaving a stale BLANK_ID or
    DO_NORMALIZE behind would silently attack the new model through the old
    model's decoding assumptions.
    """
    global model, processor, BLANK_ID, DO_NORMALIZE, MODEL_DIR, RUNG, CHECKPOINT
    ds = cfg["ckpt_dataset"]
    cands = [p for p in inputs if re.search(rf"{re.escape(ds)}$", os.path.basename(p))]
    assert len(cands) == 1, (
        f"Need exactly one input matching '{ds}', found {len(cands)}: "
        f"{[os.path.basename(c) for c in cands]}")
    ck = cands[0]
    if cfg["checkpoint"] == "final":
        MODEL_DIR = ck if os.path.exists(f"{ck}/config.json") else f"{ck}/final"
    else:
        MODEL_DIR = f"{ck}/milestones/step-{cfg['checkpoint'].split('-')[1]}"
    assert os.path.exists(f"{MODEL_DIR}/config.json"), f"no config.json under {MODEL_DIR}"

    processor = Wav2Vec2Processor.from_pretrained(MODEL_DIR)
    model = Wav2Vec2ForCTC.from_pretrained(MODEL_DIR).float().to(DEVICE).eval()
    for p in model.parameters():
        p.requires_grad_(False)
    model.config.apply_spec_augment = False
    DO_NORMALIZE = bool(getattr(processor.feature_extractor, "do_normalize", True))
    BLANK_ID = model.config.pad_token_id
    RUNG, CHECKPOINT = cfg["rung"], cfg["checkpoint"]
    print(f"  loaded {MODEL_DIR}")
    print(f"  do_normalize {DO_NORMALIZE} | blank {BLANK_ID} | vocab {model.config.vocab_size}")
    return MODEL_DIR


def unload():
    global model, processor
    try:
        del model, processor
    except NameError:
        pass
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

## 6 · The stronger attack

One function. It attacks at a single budget, from several random starting
points, and stops at the first restart that succeeds. No bisection: the question
is whether the target is reachable at the ceiling at all.

In [ ]:
import random

def strong_attack(x, mask, tgt_pad, tgt_len, targets, eps, n_restarts=N_RESTARTS,
                  steps=STRONG_STEPS):
    """Attack one batch at a single budget, with restarts. Returns per-sample
    (success, restart_index_that_worked, steps_used)."""
    B = x.shape[0]
    eps_col = torch.full((B, 1), eps, device=DEVICE)
    lr = LR_SCALE * eps

    success = np.zeros(B, dtype=bool)
    which = np.full(B, -1, dtype=int)
    used = np.zeros(B, dtype=int)
    best_delta = torch.zeros_like(x)

    for rs in range(n_restarts):
        todo = ~success
        if not todo.any():
            break
        # restart 0 starts at zero, matching the published attack; the rest
        # start uniformly inside the eps-ball.
        if rs == 0:
            delta = torch.zeros_like(x)
        else:
            delta = (torch.rand_like(x) * 2 - 1) * eps
        delta = delta.clone().requires_grad_(True)
        opt = torch.optim.Adam([delta], lr=lr)
        active = torch.tensor(todo, device=DEVICE)

        for step in range(steps):
            if not active.any():
                break
            x_adv = torch.clamp(x + delta, -1, 1) * mask
            per = ctc_per_sample(x_adv, mask, tgt_pad, tgt_len)   # fp32, no autocast
            loss = (per * active.float()).sum() / active.float().sum().clamp(min=1)
            opt.zero_grad(); loss.backward()
            if delta.grad is not None:
                delta.grad[~active] = 0
            opt.step()
            with torch.no_grad():
                delta.data = torch.max(torch.min(delta.data, eps_col), -eps_col)

            if (step + 1) % CHECK_EVERY == 0:
                hyp = decode_b(int16_rt(torch.clamp(x + delta, -1, 1)) * mask, mask)
                with torch.no_grad():
                    for i in range(B):
                        if active[i] and hyp[i] == targets[i]:
                            success[i] = True; which[i] = rs; used[i] = step + 1
                            best_delta[i] = delta[i].detach()
                            active[i] = False
        for i in range(B):
            if not success[i]:
                used[i] += steps
    return success, which, used, best_delta

## 7 · Run it on the censored utterances only

In [ ]:
def censored_ids(cfg):
    """utt_ids that the published run recorded as censored, from any mount depth."""
    hits = []
    for d in range(1, 5):
        hits += sorted(glob.glob("/kaggle/input/" + "*/" * d + cfg["source_csv"]))
    assert hits, (
        f"cannot find {cfg['source_csv']}. Attach the dataset that carries the "
        "published result for this checkpoint; without it there is no list of "
        "censored utterances to re-attack.")
    df = pd.read_csv(hits[0])
    cen = df[df.censored.astype(str).str.lower().isin(["true", "1"])]
    print(f"  {cfg['source_csv']}: {len(cen)}/{len(df)} censored  ({hits[0]})")
    return list(cen.utt_id), len(df)


SESSION_START = time.time()
DEADLINE = SESSION_START + WALL_HOURS * 3600
summary, rows_all = [], []

for qi, cfg in enumerate(QUEUE, 1):
    print("\n" + "=" * 74)
    print(f"[{qi}/{len(QUEUE)}] {cfg['tag']}   "
          f"({(DEADLINE-time.time())/3600:.1f} h left)")
    print("=" * 74)
    ids, n_total = censored_ids(cfg)
    sub_all = ev[ev.utt_id.isin(ids)].reset_index(drop=True)
    if len(sub_all) == 0:
        print("  nothing censored here; skipping.")
        summary.append(dict(tag=cfg["tag"], n_censored=0, n_flipped=0, rate=None))
        continue
    load_condition(cfg)
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

    OUT = f"{WORK}/strength_{cfg['tag']}_{EDIT_TYPE}.csv"
    rows = []
    batches = [sub_all.iloc[i:i+BATCH_SIZE] for i in range(0, len(sub_all), BATCH_SIZE)]
    for bi, sub in enumerate(batches):
        if time.time() > DEADLINE:
            print("  *** wall clock reached ***"); break
        xb, mb = load_batch(sub)
        tp = [torch.tensor(processor.tokenizer(t).input_ids, device=DEVICE)
              for t in sub.target_text]
        tl = torch.tensor([len(t) for t in tp], device=DEVICE)
        tpad = torch.nn.utils.rnn.pad_sequence(tp, batch_first=True, padding_value=0)
        t0 = time.time()
        ok, which, used, best = strong_attack(xb, mb, tpad, tl, list(sub.target_text),
                                              EPS_HI)
        dt = time.time() - t0
        x_adv = int16_rt(torch.clamp(xb + best, -1, 1)) * mb
        snrs = snr_db_b(xb, x_adv, mb)
        hyp = decode_b(x_adv, mb)
        for i, (_, r) in enumerate(sub.iterrows()):
            rows.append(dict(utt_id=r["utt_id"], tag=cfg["tag"], rung=cfg["rung"],
                             checkpoint=cfg["checkpoint"], duration=r["duration"],
                             was_censored=True, now_success=bool(ok[i]),
                             restart_used=int(which[i]), steps_used=int(used[i]),
                             eps=EPS_HI,
                             snr_db=float(snrs[i]) if ok[i] else None,
                             decoded=(hyp[i] if ok[i] else None),
                             model_dir=MODEL_DIR))
        pd.DataFrame(rows).to_csv(OUT, index=False)
        print(f"    batch {bi+1}/{len(batches)}  {dt/60:4.1f} min  "
              f"{int(ok.sum())}/{len(sub)} flipped to success")
    df = pd.DataFrame(rows)
    nf = int(df.now_success.sum()) if len(df) else 0
    rate = 100.0 * nf / len(df) if len(df) else None
    summary.append(dict(tag=cfg["tag"], n_censored=len(df), n_flipped=nf, rate=rate,
                        n_total=n_total))
    print(f"  {nf}/{len(df)} censored utterances FLIPPED "
          + (f"({rate:.0f}%)" if rate is not None else ""))
    rows_all += rows
    unload()

## 8 · What it means for eps50

In [ ]:
sm = pd.DataFrame(summary)
print(sm.to_string(index=False))

print("\n--- corrected attack success " + "-" * 44)
for r in sm.itertuples():
    if r.rate is None or not r.n_total:
        continue
    old_asr = 100.0 * (r.n_total - r.n_censored) / r.n_total
    new_asr = 100.0 * (r.n_total - r.n_censored + r.n_flipped) / r.n_total
    print(f"  {r.tag:16s} published ASR {old_asr:5.1f}%  ->  {new_asr:5.1f}% "
          f"under a stronger attack  (+{new_asr-old_asr:.1f} pp)")

if len(sm.dropna(subset=["rate"])) == 2:
    a, b = sm.dropna(subset=["rate"]).itertuples()
    gap = abs(a.rate - b.rate)
    print(f"\n  flip rate {a.tag} {a.rate:.0f}% vs {b.tag} {b.rate:.0f}%  "
          f"-> {gap:.0f} pp apart")
    if gap > 15:
        print("  THESE DIFFER A LOT. Censoring is not a fixed property of the")
        print("  measurement: it is easier to fail against one model than the other,")
        print("  so comparisons of eps50 between them carry that bias. Say so in")
        print("  the paper and treat the pair effects as upper bounds.")
    else:
        print("  Similar rates. Censoring behaves the same way at both ends of the")
        print("  ladder, so it adds noise to eps50 but does not obviously bias the")
        print("  comparison between rungs.")

if len(rows_all):
    rr = pd.DataFrame(rows_all)
    w = rr[rr.now_success]
    if len(w):
        print("\n  which restart succeeded (0 = the published zero start):")
        print(w.restart_used.value_counts().sort_index().to_string())
        print("\n  any success at restart > 0 is one the published attack could not")
        print("  have found, because it only ever started from zero.")

PROV.update(notebook=NOTEBOOK, summary=summary, use_amp=USE_AMP,
            strong_steps=STRONG_STEPS, n_restarts=N_RESTARTS, lr_scale=LR_SCALE,
            eps=EPS_HI, session_hours=round((time.time()-SESSION_START)/3600, 2))
json.dump(PROV, open(f"{WORK}/_provenance.json", "w"), ensure_ascii=False,
          indent=2, default=str)
json.dump({"title": "attack-strength-check", "id": "YOURUSERNAME/attack-strength-check",
           "licenses": [{"name": "CC0-1.0"}]},
          open(f"{WORK}/dataset-metadata.json", "w"), indent=2)
for f in sorted(os.listdir(WORK)):
    print("   ", f)
print("\nCHECK THIS FILE LIST before publishing.")